In [1]:
from pathlib import Path
import os,json,ast,hashlib,time,random
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import numpy as np,h5py,torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
from sklearn.metrics import roc_auc_score,average_precision_score
ROOT=Path(r'C:/Users/choud/Desktop/Minor project');OUT=ROOT/'fusion/cnn_bilstm_v1';V3=ROOT/'cnn/cnn wave/eew_causal_v3';CR=V3/'cnn_eew_training';BR=ROOT/'bilstm_eew/bilstm_training_v1';TAB=ROOT/'bilstm_eew/causal_tabular_v1/tabular_features.h5'
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu');torch.set_num_threads(4);torch.use_deterministic_algorithms(True);torch.backends.cudnn.benchmark=False
CFG={'seed':42,'epochs':35,'patience':7,'min_delta':1e-4,'batch_size':256,'lr':.001,'weight_decay':.0001,'dropout':.2,'clip':5.,'low_fpr_target':.05}
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(4194304),b''):h.update(b)
 return h.hexdigest()
# Import exact class definitions only: no original setup, training or evaluation executes.
for path,cell,name in [(CR/'01_cnn_training.ipynb',1,'EarlyCNN'),(BR/'01_bilstm_training.ipynb',0,'TabularBiLSTM')]:
 tree=ast.parse(''.join(json.loads(path.read_text(encoding='utf-8'))['cells'][cell]['source']));node=next(x for x in tree.body if isinstance(x,ast.ClassDef) and x.name==name);exec(compile(ast.Module(body=[node],type_ignores=[]),'original_class_only','exec'))
class FusionHead(nn.Module):
 def __init__(self):
  super().__init__();self.shared=nn.Sequential(nn.Linear(64,32),nn.SiLU(),nn.Dropout(.2));self.detection=nn.Linear(32,1);self.magnitude=nn.Linear(32,1)
 def forward(self,z):
  z=self.shared(z);return self.detection(z)[:,0],self.magnitude(z)[:,0]
def align():
 mapping={};parents={};summary={}
 with h5py.File(TAB,'r') as tab:
  for split,n in [('train',11324),('validation',2426),('test',2428)]:
   with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as w:
    t=tab[split];wp=w['parent_id'].asstr()[:];tp=t['parent_id'].asstr()[:];assert len(wp)==len(tp)==n and len(set(wp))==len(set(tp))==n,'Critical parent uniqueness/count failure'
    lookup={p:i for i,p in enumerate(tp)};assert set(wp)==set(tp),'Parent set mismatch';ix=np.array([lookup[p] for p in wp]);mapping[split]=ix;parents[split]=set(wp)
    for key in ['source_id','parent_id','pair_id','combined_index','labels','magnitude','magnitude_mask']+[f'{k}_{h}s' for h in [1,2,3] for k in ['pre_s_valid','classification_mask','magnitude_mask']]:
     a=w[key].asstr()[:] if w[key].dtype.kind=='O' else w[key][:];b=t[key].asstr()[:] if t[key].dtype.kind=='O' else t[key][:];assert np.array_equal(a,b[ix]),f'Alignment failure: {split}/{key}'
    assert w.attrs['sampling_rate_hz']==tab.attrs['sampling_rate_hz']==100 and w.attrs['pre_context_samples']==tab.attrs['pre_reference_samples']==100
    for h in [1,2,3]:
     assert w[f'windows_{h}s_norm'].shape==(n,100+h*100) and t[f'{h}s/X'].shape==(n,10+10*h,7)
     assert t[f'{h}s'].attrs['last_bin_end_relative_sample_exclusive']==h*100
     y=w['labels'][:];r=w[f'magnitude_mask_{h}s'][:];k=w[f'classification_mask_{h}s'][:];assert np.array_equal(r,(y==1)&k) and np.array_equal(r,w[f'pre_s_valid_{h}s'][:]) and np.all(w['magnitude'][:][y==0]==-1)
    summary[split]={'records':n,'reordered_tabular_rows':int((ix!=np.arange(n)).sum()),'parent_order_sha256':hashlib.sha256('\n'.join(wp).encode()).hexdigest()}
  for a,b in [('train','validation'),('train','test'),('validation','test')]:assert not(parents[a]&parents[b]),'Cross-split parent overlap'
 return mapping,summary
MAP,ALIGNMENT=align()
def branches(h):
 cc=torch.load(CR/'models'/f'cnn_{h}s_best.pt',map_location='cpu',weights_only=False);bc=torch.load(BR/'models'/f'bilstm_{h}s_best.pt',map_location='cpu',weights_only=False)
 assert cc['horizon_seconds']==bc['horizon']==h
 c=EarlyCNN().to(DEV);b=TabularBiLSTM().to(DEV);c.load_state_dict(cc['state_dict'],strict=True);b.load_state_dict(bc['state_dict'],strict=True)
 for model in [c,b]:model.requires_grad_(False);model.eval()
 return c,b,cc,bc
@torch.inference_mode()
def encode(split,h,c,b,cc,bc,only_common=False):
 with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as w,h5py.File(TAB,'r') as t:
  keep=w[f'classification_mask_{h}s'][:].astype(bool);common=np.logical_and.reduce([w[f'classification_mask_{j}s'][:] for j in [1,2,3]])
  assert np.array_equal(common,t[split]['common_cohort_mask'][:][MAP[split]])
  if only_common:keep &= common
  x=w[f'windows_{h}s_norm'][:][keep,None,:];tab=t[split][f'{h}s/X'][:][MAP[split]][keep]
  y=w['labels'][:][keep].astype('float32');m=w['magnitude'][:][keep];reg=w[f'magnitude_mask_{h}s'][:][keep];ids=w['source_id'].asstr()[:][keep]
  assert np.isfinite(x).all() and np.isfinite(tab).all()
 embeddings=[];cp=[];cm=[];bp=[];bm=[];captured={}
 hc=c.shared.register_forward_hook(lambda module,args,out:captured.update(c=out));hb=b.shared.register_forward_hook(lambda module,args,out:captured.update(b=out))
 try:
  for start in range(0,len(y),256):
   ca,cv=c(torch.from_numpy(x[start:start+256]).to(DEV));ba,bv=b(torch.from_numpy(tab[start:start+256]).to(DEV));embeddings.append(torch.cat([captured['c'],captured['b']],1).cpu());cp.append(ca.sigmoid().cpu());cm.append(cv.cpu());bp.append(ba.sigmoid().cpu());bm.append(bv.cpu())
 finally:hc.remove();hb.remove()
 z=torch.cat(embeddings);assert z.shape==(len(y),64) and not z.requires_grad
 return dict(z=z,y=torch.from_numpy(y),m=torch.from_numpy(m),reg=torch.from_numpy(reg),ids=ids,cnn_p=torch.cat(cp).numpy(),cnn_m=torch.cat(cm).numpy()*cc['target_scale']+cc['target_center'],bilstm_p=torch.cat(bp).numpy(),bilstm_m=torch.cat(bm).numpy()*bc['scale']+bc['center'])
def batches(d,mu,sd,shuffle=False):
 return DataLoader(TensorDataset((d['z']-mu)/sd,d['y'],d['m'],d['reg']),batch_size=256,shuffle=shuffle,num_workers=0,generator=torch.Generator().manual_seed(42))
def epoch(head,loader,center,scale,opt=None):
 head.train(opt is not None);ds=ms=0.;n=nr=0
 with torch.set_grad_enabled(opt is not None):
  for z,y,m,r in loader:
   z,y,m,r=[v.to(DEV) for v in [z,y,m,r]]
   if opt is not None:opt.zero_grad(set_to_none=True)
   a,b=head(z);dl=nn.functional.binary_cross_entropy_with_logits(a,y,reduction='sum');ml=nn.functional.huber_loss(b[r],(m[r]-center)/scale,delta=1.,reduction='sum') if r.any() else b.sum()*0;loss=dl/len(y)+ml/r.sum().clamp_min(1);assert torch.isfinite(loss)
   if opt is not None:loss.backward();nn.utils.clip_grad_norm_(head.parameters(),5.);opt.step()
   ds+=dl.item();ms+=ml.item();n+=len(y);nr+=int(r.sum())
 return dict(total=ds/n+ms/max(nr,1),detection=ds/n,magnitude=ms/max(nr,1))
@torch.inference_mode()
def predict(head,z,mu,sd):
 head.eval();a=[];b=[]
 for part in ((z-mu)/sd).split(256):
  u,v=head(part.to(DEV));a.append(u.sigmoid().cpu());b.append(v.cpu())
 return torch.cat(a).numpy(),torch.cat(b).numpy()
def det(y,p,t=.5):
 y=np.asarray(y)==1;q=p>=t;tp=int((y&q).sum());fp=int((~y&q).sum());tn=int((~y&~q).sum());fn=int((y&~q).sum())
 return dict(accuracy=(tp+tn)/len(y),precision=tp/(tp+fp) if tp+fp else 0.,recall=tp/(tp+fn),f1=2*tp/(2*tp+fp+fn),false_positive_rate=fp/(fp+tn),roc_auc=float(roc_auc_score(y,p)),pr_auc_ap=float(average_precision_score(y,p)),confusion_matrix=[[tn,fp],[fn,tp]])
def magnitude(a,p):
 a=a.astype(float);p=p.astype(float);e=p-a;strong=a>=5
 return dict(n=len(a),mae=float(np.abs(e).mean()),rmse=float(np.sqrt(np.mean(e**2))),r2=float(1-np.sum(e**2)/np.sum((a-a.mean())**2)),bias=float(e.mean()),actual_variance=float(a.var()),predicted_variance=float(p.var()),std_ratio=float(p.std()/a.std()),slope=float(np.polyfit(a,p,1)[0]),strong_n=int(strong.sum()),strong_underprediction_percent=float((e[strong]<0).mean()*100),strong_bias=float(e[strong].mean()))
print('Alignment PASS; 64-dimensional learned embedding concatenation; frozen backbones.')


Alignment PASS; 64-dimensional learned embedding concatenation; frozen backbones.


In [2]:
paths=[TAB,CR/'01_cnn_training.ipynb',BR/'01_bilstm_training.ipynb',CR/'final_cnn_eew_report.md',BR/'final_bilstm_eew_report.md',BR/'codex_independent_audit/audit_report.md',CR/'codex_performance_audit/cnn_performance_audit_results.json',BR/'codex_independent_audit/audit_results.json']+[V3/'dataset'/f'eew_dataset_{s}.h5' for s in ['train','validation','test']]+[base/'models'/f'{prefix}_{h}s_best.pt' for base,prefix in [(CR,'cnn'),(BR,'bilstm')] for h in [1,2,3]]
before={str(p):sha(p) for p in paths}
assert not any((OUT/'models').iterdir()),'Refuse overwrite/retraining'
for h in [1,2,3]:
 random.seed(42);np.random.seed(42);torch.manual_seed(42)
 if DEV.type=='cuda':torch.cuda.manual_seed_all(42)
 c,b,cc,bc=branches(h);tr=encode('train',h,c,b,cc,bc);va=encode('validation',h,c,b,cc,bc)
 mu=tr['z'].mean(0);sd=tr['z'].std(0,unbiased=False);sd=torch.where(sd>1e-8,sd,torch.ones_like(sd));center=float(tr['m'][tr['reg']].mean());scale=float(tr['m'][tr['reg']].std(unbiased=False))
 head=FusionHead().to(DEV);opt=torch.optim.AdamW(head.parameters(),lr=.001,weight_decay=.0001);tl=batches(tr,mu,sd,True);te=batches(tr,mu,sd);vl=batches(va,mu,sd)
 best=float('inf');stale=0;history=[];state=None;start=time.perf_counter()
 for e in range(1,36):
  epoch(head,tl,center,scale,opt);a=epoch(head,te,center,scale);v=epoch(head,vl,center,scale);history.append(dict(epoch=e,train=a,validation=v))
  if v['total']<best-.0001:best=v['total'];selected=e;stale=0;state={k:t.detach().cpu().clone() for k,t in head.state_dict().items()}
  else:stale+=1
  if e==1 or e%10==0:print(f'{h}s epoch {e}: train {a["total"]:.4f}, validation {v["total"]:.4f}',flush=True)
  if stale>=7:break
 head.load_state_dict(state);vp,vm=predict(head,va['z'],mu,sd);y=va['y'].numpy();grid=np.round(np.arange(.10,.901,.01),2);scores=[det(y,vp,t)['f1'] for t in grid];i=max(range(len(grid)),key=lambda j:(scores[j],-abs(grid[j]-.5),-grid[j]));threshold=float(grid[i])
 noise=vp[y==0];allowed=int(np.floor(.05*len(noise)));low=float(np.nextafter(np.sort(noise)[-(allowed+1)],np.float32(np.inf)));low_stats=det(y,vp,low);assert low_stats['false_positive_rate']<=.05+1e-12
 assert all(not p.requires_grad and p.grad is None for model in [c,b] for p in model.parameters())
 assert all(torch.equal(v.cpu(),cc['state_dict'][k]) for k,v in c.state_dict().items()) and all(torch.equal(v.cpu(),bc['state_dict'][k]) for k,v in b.state_dict().items()),'Backbone changed'
 ck=dict(head_state=state,horizon=h,config=CFG,embedding_mean=mu,embedding_scale=sd,target_center=center,target_scale=scale,best_epoch=selected,best_validation_loss=best,history=history,threshold=threshold,low_fpr_threshold=low,low_fpr_validation=low_stats,validation_selected=det(y,vp,threshold),training_seconds=time.perf_counter()-start,alignment=ALIGNMENT,input_hashes=before,backbones={'cnn':str(CR/'models'/f'cnn_{h}s_best.pt'),'bilstm':str(BR/'models'/f'bilstm_{h}s_best.pt')},trainable_parameters=sum(p.numel() for p in head.parameters()),environment={'torch':str(torch.__version__),'device':str(DEV),'hardware':torch.cuda.get_device_name(0) if DEV.type=='cuda' else 'CPU'},threshold_policy='Validation F1 grid .10:.01:.90; ties closest .5 then lower. Low FPR: smallest boundary above validation negative order statistic to allow <=floor(.05*Nnoise) false positives.')
 with (OUT/'models'/f'fusion_{h}s_best.pt').open('xb') as f:torch.save(ck,f)
 print(f'{h}s selected epoch {selected}/{e}; threshold {threshold:.2f}; frozen weights unchanged',flush=True)
assert all(sha(Path(p))==v for p,v in before.items())
print('All three heads selected before test inference; six original backbones preserved.')


1s epoch 1: train 0.9220, validation 0.9364
1s selected epoch 2/9; threshold 0.42; frozen weights unchanged
2s epoch 1: train 0.8265, validation 0.8429
2s epoch 10: train 0.7807, validation 0.8052
2s epoch 20: train 0.7780, validation 0.8054
2s selected epoch 15/22; threshold 0.41; frozen weights unchanged
3s epoch 1: train 0.7660, validation 0.7889
3s epoch 10: train 0.6963, validation 0.7425
3s epoch 20: train 0.6924, validation 0.7362
3s epoch 30: train 0.6912, validation 0.7358
3s selected epoch 27/34; threshold 0.36; frozen weights unchanged
All three heads selected before test inference; six original backbones preserved.
